# Lezione 12 · Pensare prima di colpire: ragionamento e test-time compute

I conti a margine della tabella delle maree, il prezzo del pensiero, il muro del caos, il voto dei cinque skipper, e il giudice.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

## La regola dei dodicesimi

*Nel racconto: la secca con un metro e ottanta d'acqua, la chiglia da due metri, il margine di trenta centimetri; le otto e mezza d'un fiato, le nove a matita.*

La marea sale di 1,20 metri in sei ore, dalle sei a mezzogiorno: 1, 2, 3, 3, 2 e 1 dodicesimo.

In [2]:
servono = 2.00 + 0.30 - 1.80                     # chiglia, più margine, meno l'acqua sulla secca
controlla("metri di marea che servono", servono, 0.50, 1e-9)

escursione = 120                                 # centimetri
dodicesimo = escursione / 12
controlla("un dodicesimo (cm)", dodicesimo, 10, 0)

regola = np.cumsum([1, 2, 3, 3, 2, 1]) * dodicesimo
for ora, cm, atteso in zip(range(7, 13), regola, [10, 30, 60, 90, 110, 120]):
    controlla(f"regola, ore {ora} (cm)", cm, atteso, 0)

✓  metri di marea che servono: calcolato 0.5, nel libro 0.5
✓  un dodicesimo (cm): calcolato 10, nel libro 10
✓  regola, ore 7 (cm): calcolato 10, nel libro 10
✓  regola, ore 8 (cm): calcolato 30, nel libro 30
✓  regola, ore 9 (cm): calcolato 60, nel libro 60
✓  regola, ore 10 (cm): calcolato 90, nel libro 90
✓  regola, ore 11 (cm): calcolato 110, nel libro 110
✓  regola, ore 12 (cm): calcolato 120, nel libro 120


Il conto d'un fiato tratta la salita come una retta. A matita, invece: alle otto e mezza la marea è a metà della terza ora.

In [3]:
controlla("conto d'un fiato (ore dopo le sei)", servono * 100 / escursione * 6, 2.5, 1e-9)

alle_otto_e_mezza = regola[1] + 0.5 * 3 * dodicesimo
controlla("salita alle 8.30 (cm)", alle_otto_e_mezza, 45, 0)
controlla("alle nove, centimetri in più del margine", regola[2] - servono * 100, 10, 1e-9)

✓  conto d'un fiato (ore dopo le sei): calcolato 2.5, nel libro 2.5
✓  salita alle 8.30 (cm): calcolato 45, nel libro 45
✓  alle nove, centimetri in più del margine: calcolato 10, nel libro 10


La regola approssima una sinusoide. Ora per ora, in centimetri saliti dalle sei: la regola, la marea vera e il conto d'un fiato.

In [4]:
ore = np.arange(1, 7)
vera = escursione / 2 * (1 - np.cos(np.pi * ore / 6))
retta = escursione / 6 * ore

controlla("marea vera a fine della prima ora (% dell'escursione)", vera[0] / escursione * 100, 6.7, 0.05)
controlla("regola a fine della prima ora (% dell'escursione)", regola[0] / escursione * 100, 8.3, 0.05)
controlla("marea vera a fine della terza ora (% dell'escursione)", vera[2] / escursione * 100, 50, 1e-9)

print()
tabella = {7: (10, 8, 20), 8: (30, 30, 40), 9: (60, 60, 60), 10: (90, 90, 80), 11: (110, 112, 100), 12: (120, 120, 120)}
for i, ora in enumerate(range(7, 13)):
    r, v, f = tabella[ora]
    controlla(f"ore {ora}: regola", regola[i], r, 0)
    controlla(f"ore {ora}: marea vera", vera[i], v, 0.5)
    controlla(f"ore {ora}: conto d'un fiato", retta[i], f, 1e-9)

print()
controlla("errore massimo della regola, ora per ora (cm)", np.abs(regola - vera).max(), 2, 0.05)
print(f"errore massimo della retta, ora per ora: {np.abs(retta - vera).max():.0f} cm")

✓  marea vera a fine della prima ora (% dell'escursione): calcolato 6.699, nel libro 6.7
✓  regola a fine della prima ora (% dell'escursione): calcolato 8.333, nel libro 8.3
✓  marea vera a fine della terza ora (% dell'escursione): calcolato 50, nel libro 50

✓  ore 7: regola: calcolato 10, nel libro 10
✓  ore 7: marea vera: calcolato 8.038, nel libro 8
✓  ore 7: conto d'un fiato: calcolato 20, nel libro 20
✓  ore 8: regola: calcolato 30, nel libro 30
✓  ore 8: marea vera: calcolato 30, nel libro 30
✓  ore 8: conto d'un fiato: calcolato 40, nel libro 40
✓  ore 9: regola: calcolato 60, nel libro 60
✓  ore 9: marea vera: calcolato 60, nel libro 60
✓  ore 9: conto d'un fiato: calcolato 60, nel libro 60
✓  ore 10: regola: calcolato 90, nel libro 90
✓  ore 10: marea vera: calcolato 90, nel libro 90
✓  ore 10: conto d'un fiato: calcolato 80, nel libro 80
✓  ore 11: regola: calcolato 110, nel libro 110
✓  ore 11: marea vera: calcolato 112, nel libro 112
✓  ore 11: conto d'un fiato: calcolato 

Quando arrivano i 50 centimetri: per la retta alle 8.30, per la regola alle 8.40.

In [5]:
def orario(ore_dopo_le_sei):
    minuti = round(ore_dopo_le_sei * 60)
    return f"{6 + minuti // 60}.{minuti % 60:02d}"

ore_retta = 50 / (escursione / 6)
ore_regola = 2 + (50 - regola[1]) / (3 * dodicesimo)       # nella terza ora sale di tre dodicesimi
ore_vera = 6 / np.pi * math.acos(1 - 2 * 50 / escursione)

print(f"retta: {orario(ore_retta)}   regola: {orario(ore_regola)}   marea vera: {orario(ore_vera)}")
assert orario(ore_retta) == "8.30" and orario(ore_regola) == "8.40"
print("✓  come nel libro: la retta alle 8.30, la regola alle 8.40")

retta: 8.30   regola: 8.40   marea vera: 8.41
✓  come nel libro: la retta alle 8.30, la regola alle 8.40


## Il pensiero si paga

*Nel racconto: «l'unico modo che ha per pensare di più è scrivere di più».*

Con il modello da 70 miliardi della lezione precedente, a una ventina di token al secondo. Ogni token aggiunge alla busta 327.680 byte.

In [6]:
token_al_secondo = 20
byte_per_token = 2 * 80 * 8 * 128 * 2            # la busta, come per la nave di linea
conta("byte per token", byte_per_token, 327_680)

controlla("5.000 token di pensiero: minuti di attesa", 5000 / token_al_secondo / 60, 4, 0.2)
controlla("5.000 token di pensiero: GB di busta", 5000 * byte_per_token / 1e9, 1.6, 0.05)

✓  byte per token: calcolato 327.680, nel libro 327.680
✓  5.000 token di pensiero: minuti di attesa: calcolato 4.167, nel libro 4
✓  5.000 token di pensiero: GB di busta: calcolato 1.638, nel libro 1.6


La stessa risposta di 400 token, con e senza ragionamento, a 15 euro per milione di token scritti.

In [7]:
prezzo_scritto = 15                              # euro per milione di token scritti

def riga(token):
    attesa = token / token_al_secondo                          # secondi
    costo = token * prezzo_scritto / 1e6 * 100                 # centesimi
    busta = token * byte_per_token / 1e9                       # GB
    return attesa, costo, busta

attesa, costo, busta = riga(400)
controlla("senza ragionamento: attesa (secondi)", attesa, 20, 0)
controlla("senza ragionamento: costo (centesimi)", costo, 0.6, 1e-9)
controlla("senza ragionamento: busta (GB)", busta, 0.13, 0.005)

attesa_p, costo_p, busta_p = riga(400 + 5000)
controlla("con 5.000 token di pensiero: attesa (minuti)", attesa_p / 60, 4.5, 1e-9)
controlla("con 5.000 token di pensiero: costo (centesimi)", costo_p, 8, 0.1)
controlla("con 5.000 token di pensiero: busta (GB)", busta_p, 1.8, 0.05)

controlla("quante volte di più, in tutte e tre le righe: «tredici volte e mezza»", (400 + 5000) / 400, 13.5, 0.01)
print(f"e cinque tentativi costano cinque volte tutto: {5 * costo_p:.0f} centesimi, {5 * attesa_p / 60:.1f} minuti di scrittura")

✓  senza ragionamento: attesa (secondi): calcolato 20, nel libro 20
✓  senza ragionamento: costo (centesimi): calcolato 0.6, nel libro 0.6
✓  senza ragionamento: busta (GB): calcolato 0.1311, nel libro 0.13
✓  con 5.000 token di pensiero: attesa (minuti): calcolato 4.5, nel libro 4.5
✓  con 5.000 token di pensiero: costo (centesimi): calcolato 8.1, nel libro 8
✓  con 5.000 token di pensiero: busta (GB): calcolato 1.769, nel libro 1.8
✓  quante volte di più, in tutte e tre le righe: «tredici volte e mezza»: calcolato 13.5, nel libro 13.5
e cinque tentativi costano cinque volte tutto: 40 centesimi, 22.5 minuti di scrittura


## Il muro del caos

*Nel racconto: il vento che nessuno prevede oltre un paio di settimane; Edward Lorenz e il numero arrotondato.*

Lorenz ripartì da 0,506 invece di 0,506127. Se l'errore raddoppia ogni due giorni, un errore dell'1% sulla misura di oggi cresce così.

In [8]:
print(f"il numero arrotondato di Lorenz: una differenza di {0.506127 - 0.506:.6f}, il {(0.506127 - 0.506) / 0.506127:.3%}")

errore = 1                                       # per cento, oggi
for giorno, atteso in zip(range(2, 15, 2), [2, 4, 8, 16, 32, 64, 128]):
    errore *= 2
    controlla(f"giorno {giorno}: errore (%)", errore, atteso, 0)

# per spostare il muro di g giorni, l'errore iniziale va diviso per 2^(g/2)
controlla("spostare il muro di due giorni: dividere l'errore per", 2 ** (2 / 2), 2, 0)
controlla("spostare il muro di una settimana: dividere l'errore per", 2 ** (7 / 2), 11, 0.5)

il numero arrotondato di Lorenz: una differenza di 0.000127, il 0.025%
✓  giorno 2: errore (%): calcolato 2, nel libro 2
✓  giorno 4: errore (%): calcolato 4, nel libro 4
✓  giorno 6: errore (%): calcolato 8, nel libro 8
✓  giorno 8: errore (%): calcolato 16, nel libro 16
✓  giorno 10: errore (%): calcolato 32, nel libro 32
✓  giorno 12: errore (%): calcolato 64, nel libro 64
✓  giorno 14: errore (%): calcolato 128, nel libro 128
✓  spostare il muro di due giorni: dividere l'errore per: calcolato 2, nel libro 2
✓  spostare il muro di una settimana: dividere l'errore per: calcolato 11.31, nel libro 11


## Il voto

*Nel racconto: sei su dieci che diventa il sessantotto per cento, quattro su dieci che diventa il trentadue.*

La maggioranza di cinque ha ragione quando almeno tre tentativi sono giusti: P = p⁵ + 5 × p⁴ × (1 − p) + 10 × p³ × (1 − p)².

In [9]:
def maggioranza(p, n=5):
    return sum(math.comb(n, k) * p**k * (1 - p)**(n - k) for k in range(n // 2 + 1, n + 1))

p = 0.6
controlla("cinque giusti su cinque", p**5, 0.078, 0.0005)
controlla("quattro giusti su cinque", 5 * p**4 * (1 - p), 0.259, 0.0005)
controlla("tre giusti su cinque", 10 * p**3 * (1 - p)**2, 0.346, 0.0005)
controlla("p = 0,6", maggioranza(0.6), 0.683, 0.0005)
controlla("p = 0,4", maggioranza(0.4), 0.317, 0.0005)

✓  cinque giusti su cinque: calcolato 0.07776, nel libro 0.078
✓  quattro giusti su cinque: calcolato 0.2592, nel libro 0.259
✓  tre giusti su cinque: calcolato 0.3456, nel libro 0.346
✓  p = 0,6: calcolato 0.6826, nel libro 0.683
✓  p = 0,4: calcolato 0.3174, nel libro 0.317


L'effetto cresce con il numero delle voci, sempre con errori indipendenti e due sole risposte.

In [10]:
for voti, sei, quattro in [(1, 60, 40), (3, 65, 35), (5, 68, 32), (11, 75, 25)]:
    controlla(f"{voti} voti, chi ha ragione 6 volte su 10 (%)", maggioranza(0.6, voti) * 100, sei, 0.5)
    controlla(f"{voti} voti, chi ha ragione 4 volte su 10 (%)", maggioranza(0.4, voti) * 100, quattro, 0.5)

✓  1 voti, chi ha ragione 6 volte su 10 (%): calcolato 60, nel libro 60
✓  1 voti, chi ha ragione 4 volte su 10 (%): calcolato 40, nel libro 40
✓  3 voti, chi ha ragione 6 volte su 10 (%): calcolato 64.8, nel libro 65
✓  3 voti, chi ha ragione 4 volte su 10 (%): calcolato 35.2, nel libro 35
✓  5 voti, chi ha ragione 6 volte su 10 (%): calcolato 68.26, nel libro 68
✓  5 voti, chi ha ragione 4 volte su 10 (%): calcolato 31.74, nel libro 32
✓  11 voti, chi ha ragione 6 volte su 10 (%): calcolato 75.35, nel libro 75
✓  11 voti, chi ha ragione 4 volte su 10 (%): calcolato 24.65, nel libro 25


## La regata e il giudice

*Nel racconto: i giudici in acqua; Leslie Lamport e i generali bizantini.*

Cinque tentativi indipendenti, ciascuno giusto 4 volte su 10. Un giudice che riconosce sempre la risposta giusta ha bisogno solo che almeno una delle cinque lo sia.

In [11]:
controlla("con il voto", maggioranza(0.4), 0.32, 0.005)
controlla("con il giudice, 1 − 0,6⁵", 1 - 0.6**5, 0.92, 0.005)

✓  con il voto: calcolato 0.3174, nel libro 0.32
✓  con il giudice, 1 − 0,6⁵: calcolato 0.9222, nel libro 0.92


I generali bizantini: con messaggi che non si possono firmare, i generali devono essere almeno 3m + 1, dove m è il numero dei traditori.

In [12]:
for generali, traditori in [(4, 1), (7, 2), (10, 3)]:
    conta(f"con {generali} generali, traditori che si reggono", (generali - 1) // 3, traditori)
    assert generali >= 3 * traditori + 1 and traditori < generali / 3     # meno di un terzo

✓  con 4 generali, traditori che si reggono: calcolato 1, nel libro 1
✓  con 7 generali, traditori che si reggono: calcolato 2, nel libro 2
✓  con 10 generali, traditori che si reggono: calcolato 3, nel libro 3


## Kata per casa · Esercizio 1

Cinque tentativi allo stesso problema danno 42, 42, 41, 42, 40. Quale risposta scegli con il voto?

In [13]:
from collections import Counter
voti = Counter([42, 42, 41, 42, 40])
print(voti.most_common())
risposta, quanti = voti.most_common(1)[0]
conta("la risposta scelta con il voto", risposta, 42)
conta("voti su cinque", quanti, 3)

[(42, 3), (41, 1), (40, 1)]
✓  la risposta scelta con il voto: calcolato 42, nel libro 42
✓  voti su cinque: calcolato 3, nel libro 3
